# CellGPS spatial topology from a decoded transcript table

This notebook applies CellGPS/COSTE directly to decoded ISS transcript points. Each gene is treated as a labelled spatial entity, so the result is a **gene-distribution StructureMap**. It is segmentation-free and does not produce cells, masks, or cell-type topology.

Keep this notebook on the `ISS_postprocessing` kernel. Create the isolated CPU-only CellGPS runtime once from the repository root:

```bash
conda env create -f ISS_postprocessing/environments/cellgps.yml
```

No GPU is required.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import ISS_postprocessing.spatial_topology as TOPOLOGY

## Select the decoded table

The defaults match the other transcript-first ISS adapters: gene names in `target` and coordinates in `xc`/`yc`. CSV, CSV.GZ, TSV, and Parquet are accepted. Add a quality column and threshold below if appropriate for your decoder.

In [ ]:
input_dir = Path('/path/to/experiment')
region = 'R1'
coordinate_unit = 'pixel'  # use 'um' if xc/yc are already micrometres
transcripts = input_dir / region / 'decoding' / '2_decoded' / f'{region}_decoded.csv'

spots = pd.read_csv(transcripts)
display(spots.head())
print(f"{len(spots):,} transcripts; {spots['target'].nunique():,} genes")

In [ ]:
gene_counts = spots['target'].value_counts()
display(gene_counts.describe().to_frame('transcripts_per_gene'))
display(gene_counts.head(20).rename('n_transcripts').to_frame())

sample = spots.sample(min(len(spots), 100_000), random_state=0)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(sample['xc'], sample['yc'], s=0.2, alpha=0.35)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_title(f'{region}: decoded transcript density')
ax.set_xlabel('x')
ax.set_ylabel('y');

## Validate and prepare without running CellGPS

Low-count genes give unstable nearest-neighbour summaries, so choose `min_transcripts_per_gene` after inspecting the counts above. `genes=None` keeps every gene passing that threshold; pass an explicit gene list for a focused panel. Preparation writes normalized `x`, `y`, and `celltype` columns plus a complete runner configuration.

In [ ]:
parameters = dict(
    gene_col='target',
    x_col='xc',
    y_col='yc',
    quality_col=None,        # e.g. 'quality_minimum'
    min_quality=None,        # e.g. 0.5
    genes=None,              # or ['GeneA', 'GeneB', ...]
    min_transcripts_per_gene=5,
    linkage_method='average',
    batch_size=100_000,
    show_corr=True,
)

prepared = TOPOLOGY.prepare_cellgps_from_transcripts(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    **parameters,
)
points = pd.read_csv(prepared['points'])
display(points.head())
display(points['celltype'].value_counts().rename('n_transcripts').to_frame())
print(json.dumps(json.loads(prepared['config'].read_text()), indent=2))

## Run CellGPS/COSTE

The memory-optimized implementation processes nearest-neighbour queries in batches. Lower `batch_size` if memory is constrained. Set `overwrite=True` after changing parameters; otherwise a completed result is reused.

In [ ]:
outputs = TOPOLOGY.run_cellgps_from_transcripts(
    transcripts,
    input_dir=input_dir,
    region=region,
    coordinate_unit=coordinate_unit,
    cellgps_python=['conda', 'run', '-n', 'cellgps', 'python'],
    overwrite=False,
    **parameters,
)
outputs

## Inspect the gene-distribution StructureMaps

Rows are Searcher genes and columns are Findee genes in the directed nearest-neighbour calculation. The two cophenetic matrices capture different sides of that directed topology and need not match. Values are normalized topology distances—not co-expression, differential expression, or physical micrometres.

In [ ]:
searcher = pd.read_csv(outputs['searcher_matrix'], index_col=0)
findee = pd.read_csv(outputs['findee_matrix'], index_col=0)
metadata = json.loads(outputs['metadata'].read_text())

display(pd.read_csv(outputs['entity_counts']))
print(json.dumps(metadata, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
sns.heatmap(searcher, cmap='RdBu', vmin=0, vmax=1, square=True, ax=axes[0])
axes[0].set_title('Searcher gene StructureMap')
sns.heatmap(findee, cmap='RdBu', vmin=0, vmax=1, square=True, ax=axes[1])
axes[1].set_title('Findee gene StructureMap')
plt.tight_layout();

## Recommended checks

- Repeat the run with a higher `min_transcripts_per_gene`; sparse genes can be unstable.
- Inspect spatial plots of genes that cluster together before assigning biological meaning.
- Compare results across fields of view or biological replicates instead of interpreting a single hierarchy in isolation.
- CellGPS captures spatial topology, while SAINSC and SFUMATO answer different questions (density maxima/cell proxies and spatial domains, respectively).